In [ ]:
## 2D histogrammer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import os
from tqdm import tqdm

mapdf = pd.read_csv('/home/kale-chen/Documents/PET/TPPT_Scanner_map_adjusted.csv', header = None)
mapL = mapdf.iloc[3072:]
mapR = mapdf.iloc[:3072]
mapL = np.round(mapL, 2)
mapR = np.round(mapR, 2)
ycoordsL, countsL = np.unique(mapL.iloc[:, 1], return_counts=True)
zcoordsL, countsL = np.unique(mapL.iloc[:, 2], return_counts=True)
ycoordsR, countsR = np.unique(mapR.iloc[:, 1], return_counts=True)
ycoordsR = ycoordsR[::-1] # To keep view from isocenter outwards
zcoordsR, countsR = np.unique(mapR.iloc[:, 2], return_counts=True)
scanner_arc_L = np.zeros((32, 96))
scanner_arc_R = np.zeros((32, 96))
for i in range(3072):
    scanner_arc_R[np.where(zcoordsR == mapR.iloc[i, 2]), np.where(ycoordsR == mapR.iloc[i, 1])] = i
for i in range(3072):
    scanner_arc_L[np.where(zcoordsL == mapL.iloc[i, 2]), np.where(ycoordsL == mapL.iloc[i, 1])] = i+3072
def loadmap(mapver = 0):
    if mapver == 0:
        mapdf = pd.read_csv('/home/kale-chen/Documents/PET/TPPT_Scanner_map_adjusted.csv', header = None)
    elif mapver == 1:
        mapdf = pd.read_csv('/home/kale-chen/Documents/PET/LUT.txt', sep = ' ', header = None)
    mapL = mapdf.iloc[3072:]
    mapR = mapdf.iloc[:3072]
    mapL = np.round(mapL, 2)
    mapR = np.round(mapR, 2)
    ycoordsL, countsL = np.unique(mapL.iloc[:, 1], return_counts=True)
    zcoordsL, countsL = np.unique(mapL.iloc[:, 2], return_counts=True)
    ycoordsR, countsR = np.unique(mapR.iloc[:, 1], return_counts=True)
    ycoordsR = ycoordsR[::-1] # To keep view from isocenter outwards
    zcoordsR, countsR = np.unique(mapR.iloc[:, 2], return_counts=True)
    scanner_arc_L = np.zeros((32, 96))
    scanner_arc_R = np.zeros((32, 96))
    for i in range(3072):
        scanner_arc_R[np.where(zcoordsR == mapR.iloc[i, 2]), np.where(ycoordsR == mapR.iloc[i, 1])] = i
    for i in range(3072):
        scanner_arc_L[np.where(zcoordsL == mapL.iloc[i, 2]), np.where(ycoordsL == mapL.iloc[i, 1])] = i+3072



heatmapmin, heatmapmax = 0, 1
def getcolor(value, cmap = 'RdBu', numdiscretecolors = 20):
    global heatmapmin, heatmapmax
    colormap = plt.get_cmap(cmap)
    if value < heatmapmin:
        return colormap(0.0)
    elif value > heatmapmax:
        return colormap(1.0)
    else:
        return colormap((value - heatmapmin) / (heatmapmax - heatmapmin) * (numdiscretecolors - 2) // 1 / (numdiscretecolors - 2))
def getcolornondiscrete(value, cmap = 'RdBu'):
    global heatmapmin, heatmapmax
    colormap = plt.get_cmap(cmap)
    if value < heatmapmin:
        return colormap(0.0)
    elif value > heatmapmax:
        return colormap(1.0)
    else:
        return colormap((value - heatmapmin) / (heatmapmax - heatmapmin))


def makeheatmap(heatmapL, heatmapR, title = '', cbarlabel = '', savedir = None, deadcondition = lambda x: x == 0):
    for i in range(32):
        for j in range(96):
            if deadcondition(heatmapR[i, j]):
                heatmapR[i, j] = -1
            if deadcondition(heatmapL[i, j]):
                heatmapL[i, j] = -1
    blank_color = 'black'
    cmap = plt.get_cmap('RdBu_r') # 'seismic' or 'RdBu' or 'rwb'
    fig, ax = plt.subplots(figsize = (8, 5.4))
    for z in range(32):
        zspace = z // 8 * 1.2
        for y in range(96):
            yspace = y // 8 * 1.2
            if heatmapR[z, y] != -1:
                rectR = patches.Rectangle((y*3.2 + yspace, -106 + z*3.2 + zspace - 2.135), 3.2, 3.2, facecolor=getcolor(heatmapR[z, y], cmap = cmap))
            else:
                rectR = patches.Rectangle((y*3.2 + yspace, -106 + z*3.2 + zspace - 2.135), 3.2, 3.2, facecolor=blank_color)
            ax = plt.gca()
            ax.add_patch(rectR)
            if heatmapL[z, y] != -1:
                rectL = patches.Rectangle((y*3.2 + yspace, -106 + z*3.2 + zspace + 108.135), 3.2, 3.2, facecolor=getcolor(heatmapL[z, y], cmap = cmap))
            else:
                rectL = patches.Rectangle((y*3.2 + yspace, -106 +z*3.2 + zspace + 108.135), 3.2, 3.2, facecolor=blank_color)
            ax.add_patch(rectL)

    bounds = np.linspace(heatmapmin, heatmapmax, 21)
    norm = colors.BoundaryNorm(bounds, cmap.N)
    mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = plt.colorbar(mappable, ax=ax, orientation='vertical', fraction=0.02, aspect = 40, pad=0.01)
    #diff = heatmapmax - heatmapmin
    #cbar.set_ticks([heatmapmin, heatmapmin+diff*0.05, heatmapmin+diff*0.1, heatmapmin+diff*0.15, heatmapmin+diff*0.2, heatmapmin+diff*0.25, heatmapmin+diff*0.3, heatmapmin+diff*0.35, heatmapmin+diff*0.4, heatmapmin+diff*0.45, heatmapmin+diff*0.5, heatmapmin+diff*0.55, heatmapmin+diff*0.6, heatmapmin+diff*0.65, heatmapmin+diff*0.7, heatmapmin+diff*0.75, heatmapmin+diff*0.8, heatmapmin+diff*0.85, heatmapmin+diff*0.9, heatmapmin+diff*0.95, heatmapmax])

    formatter = ticker.ScalarFormatter(useMathText=True)

    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))
    cbar.ax.yaxis.offsetText.set_visible(True)  # Make sure the offset (e.g., x1e3) is shown
    cbar.ax.yaxis.get_offset_text().set_position((8, 10))  # Move offset (like 1e3) to the right of colorbar
    cbar.ax.yaxis.get_offset_text().set_fontsize(8)
    cbar.formatter = formatter
    cbar.update_ticks()
    cbar.set_ticks([heatmapmin, 
                    heatmapmin + (heatmapmax-heatmapmin)*0.25, 
                    heatmapmin + (heatmapmax-heatmapmin)*0.5, 
                    heatmapmin + (heatmapmax-heatmapmin)*0.75, 
                    heatmapmax])
    cbar.ax.tick_params(labelsize=8)
    cbar.set_label(cbarlabel, rotation=270, labelpad=10, fontsize=8)
    plt.xlim(0,320.4)
    plt.ylim(-108.135, 108.135)
    plt.axis('off')
    #plt.tight_layout()
    plt.title(title, fontsize=10)
    plt.text(-5, 55, 'Left', ha='center', va='center', fontsize=8, color='black', rotation=90)
    plt.text(-5, -55, 'Right', ha='center', va='center', fontsize=8, color='black', rotation=90)
    if savedir:
        plt.savefig(os.path.join(savedir, f'{title}.pdf'), format='pdf', dpi=300, bbox_inches='tight')
    plt.show()

def radialaxialhist(eventcounts):
    radialL = np.sort(np.unique(np.round((np.arctan2(mapL.iloc[:, 1], mapL.iloc[:, 0]) * 180 / np.pi) % 360 - 180, 1))) 
    radialR = np.sort(np.unique(np.round((np.arctan2(mapR.iloc[:, 1], mapR.iloc[:, 0]) * 180 / np.pi + 180) % 360 - 180, 1)))
    axialL = np.unique(np.round(mapL.iloc[:, 2], 1))
    axialR = np.unique(np.round(mapR.iloc[:, 2], 1))
    axialheightsL, axialheightsR = np.zeros(len(axialL)), np.zeros(len(axialR))
    radialheightsL, radialheightsR = np.zeros(len(radialL)), np.zeros(len(radialR))
    for i in range(3072):
        index = np.where(radialL == np.round(np.arctan2(mapL.iloc[i, 1], mapL.iloc[i, 0]) * 180 / np.pi % 360 - 180, 1))
        radialheightsL[index] += eventcounts[i + 3072]
        index = np.where(axialL == np.round(mapL.iloc[i, 2], 1))
        axialheightsL[index] += eventcounts[i + 3072]
    for i in range(3072):
        index = np.where(radialR == np.round((np.arctan2(mapR.iloc[i, 1], mapR.iloc[i, 0]) * 180 / np.pi + 180) % 360 - 180, 1))
        radialheightsR[index] += eventcounts[i]
        index = np.where(axialR == np.round(mapR.iloc[i, 2], 1))
        axialheightsR[index] += eventcounts[i]
    return radialL, radialR, axialL, axialR, radialheightsL, radialheightsR, axialheightsL, axialheightsR

